In [1]:
#/root/jupyter_notebooks/mvm-accelerator/main.ipynb

import sys, os, time

import numpy as np
import scipy.io as sio
from pathlib import Path   

model_dir = '/home/ubuntu/mvm-accelerator/sw/model' 

if model_dir not in sys.path:
    sys.path.append(model_dir) 

from init import initialize
from results import results
from backward import backward
from plots import plots

In [2]:
OFFLOAD     = True                # True <=> use RTL logic for MVM
SMALL_MODEL = True                # True <=> N=168 else N=17048
M_PRECISION = 64                  # <16, 32, 64>
K_PRECISION = 16                  # <16, 32, 64>
PRESCALE    = (K_PRECISION == 16) # Pre-scale matrix for low-precision range

DATA_DIR = "Data_Country" if SMALL_MODEL else "Data"
NPAD     = 192 if SMALL_MODEL else 17088 # Matrix dimensions with padding

m_dtype = {16: np.float16, 32: np.float32, 64: np.float64}[M_PRECISION]
k_dtype = {16: np.float16, 32: np.float32, 64: np.float64}[K_PRECISION]

# Initialize model
state = initialize(M_PRECISION, data_dir=DATA_DIR, prescale=PRESCALE)

if OFFLOAD:
    from kernel import MVMKernel
    state.Npad = NPAD
    state.kernel = MVMKernel(
        bitfile=os.path.join(model_dir, f"../../hw/fp{K_PRECISION}_{NPAD}x{NPAD}_250/design_1.bit"),
        matrix=state.trmult_reduced_padded,
        Npad=NPAD,
        element_width_bits=K_PRECISION,
        file_type="npy",
        verbose=1
)

# Distribution of land for simulation
H0_arr = np.asarray(state.H0).reshape(-1)
H = H0_arr[state.earth_indices]

# Number of periods
nb_per = 600

# Number of periods for backward simulation
nb_back = 180

[kernel] dtype: <class 'numpy.float16'>
[kernel] Npad: 192
[kernel] channels: 4
[kernel] rows_per_channel: 48
[kernel] elements_per_row: 192
[kernel] elements_per_partition: 48
[kernel] bytes_per_partition: 96
[kernel] partition_base: ['0x80000000', '0x80000080', '0x80000100', '0x80000180']
[kernel] matrix CMA allocation OK: tile_rows=48
[kernel] initialization done: TILE_MODE=False


In [3]:
# Warm-up
for i in range(4):
    t = time.perf_counter()
    res = state.mvm(np.ones(NPAD))
    print(time.perf_counter() - t)
    print(res)

0.05210011900635436
[1.55761719e-01 1.27792358e-02 3.00025940e-03 2.70703125e+00
 6.97021484e-02 6.15234375e-02 3.94287109e-02 3.13476562e-01
 1.44195557e-02 2.61718750e+00 1.83258057e-02 1.49078369e-02
 5.65429688e-01 1.86889648e-01 9.39941406e-02 1.32217407e-02
 2.95829773e-03 2.90222168e-02 2.59780884e-03 1.08032227e-02
 2.56347656e-01 3.51562500e-02 4.29153442e-03 2.19879150e-02
 1.57165527e-02 9.06372070e-03 1.63879395e-02 6.44531250e-01
 2.46047974e-03 1.88446045e-03 2.16979980e-02 1.59454346e-03
 7.21359253e-03 1.14453125e+00 2.52532959e-02 1.01074219e-01
 7.27462769e-03 1.72363281e-01 6.90429688e-01 2.82714844e-01
 1.40953064e-03 6.61621094e-01 1.79199219e-01 3.86962891e-02
 1.29852295e-02 1.94824219e-01 3.40576172e-01 1.48437500e-01
 3.79562378e-04 2.16865540e-03 1.24931335e-04 0.00000000e+00
 6.78300858e-05 1.35231018e-03 2.81219482e-02 8.75854492e-03
 1.79481506e-03 3.09753418e-02 3.16238403e-03 7.28988647e-03
 0.00000000e+00 8.25500488e-03 1.38015747e-02 5.38825989e-04
 1.1

In [5]:
start = time.perf_counter()

# Run the model and obtain summary statistics
results_data = results(H, nb_per, state, m_dtype, prescaled=PRESCALE)
print(f"model elapsed time: {(time.perf_counter() - start) / 60:.3f} minutes")

realgdp_w, u_w, u2_w, prod_w, phi_w, PDV_u_w, PDV_u2_w, PDV_realgdp_w, migr_cell, migr_ctry, l, u, u2, tau, realgdp = results_data

t=1
iter 1: error=76979006620319.44
iter 2: error=112804145273.33241
iter 3: error=28157418538.49705
iter 4: error=7038199720.602564
iter 5: error=1759451123.2796133
iter 6: error=439850754.4861436
iter 7: error=109959741.05007268
iter 8: error=27489935.262520466
iter 9: error=6872483.81563024
iter 10: error=1718120.9539074204
iter 11: error=429530.2384769416
iter 12: error=107382.55961931396
iter 13: error=26845.639904772164
iter 14: error=6711.409976196535
iter 15: error=1677.8524940330929
iter 16: error=419.46312351614347
iter 17: error=104.86578087320453
iter 18: error=26.216445221210087


/home/ubuntu/mvm-accelerator/sw/model/model.py:107: RuntimeWarning: divide by zero encountered in power
  input_integral_inner = input_integral_outer * uhat_loop ** i_exp


iter 19: error=6.554111306266731
iter 20: error=1.6385278259712401
iter 21: error=0.40963195629889043
iter 22: error=0.10240798920741238
iter 23: error=0.025601997367030713
iter 24: error=0.006400499303371913
ile elapsed time (t = 0): 0.2711 (s)
num_iterations: 24
 1%: 0.0
10%: 0.0
25%: 2617.569233771559
50%: 24563.787274193586
75%: 271554.95369961206
90%: 1055201.6724497243
99%: 2652046.3359697754
uhat_loop avg: 295272.2775495133


SystemExit: -1

/usr/local/share/pynq-venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3406: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
start = time.perf_counter()

# Plot time series and maps, and save them
plots(H, realgdp_w, u_w, u2_w, prod_w, l, u, tau, realgdp)
print(f"plots elapsed time: {(time.perf_counter() - start) / 60:.3f} minutes")

In [6]:
start = time.perf_counter()

# Run model backwards
l_b, u_b, w_b, tau_b, phi_b, realgdp_b = backward(H, nb_back, state, m_dtype, prescaled=PRESCALE)
print(f"backward elapsed time: {(time.perf_counter() - start) / 60:.3f} minutes")

t=-1


/home/ubuntu/mvm-accelerator/sw/model/backward.py:112: RuntimeWarning: divide by zero encountered in power
  input_integral_inner = input_integral_outer * l_loop ** i_exp


t=-2
t=-3
t=-4
t=-5
t=-6
t=-7
t=-8
t=-9
t=-10
t=-11
t=-12
t=-13
t=-14
t=-15
t=-16
t=-17
t=-18
t=-19
t=-20
t=-21
t=-22
t=-23
t=-24
t=-25
t=-26
t=-27
t=-28
t=-29
t=-30
t=-31
t=-32
t=-33
t=-34
t=-35
t=-36
t=-37
t=-38
t=-39
t=-40
t=-41
t=-42
t=-43
t=-44
t=-45
t=-46
t=-47
t=-48
t=-49
t=-50
t=-51
t=-52
t=-53
t=-54
t=-55
t=-56
t=-57
t=-58
t=-59
t=-60
t=-61
t=-62
t=-63
t=-64
t=-65
t=-66
t=-67
t=-68
t=-69
t=-70
t=-71
t=-72
t=-73
t=-74
t=-75
t=-76
t=-77
t=-78
t=-79
t=-80
t=-81
t=-82
t=-83
t=-84
t=-85
t=-86
t=-87
t=-88
t=-89
t=-90
t=-91
t=-92
t=-93
t=-94
t=-95
t=-96
t=-97
t=-98
t=-99
t=-100
t=-101
t=-102
t=-103
t=-104
t=-105
t=-106
t=-107
t=-108
t=-109
t=-110
t=-111
t=-112
t=-113
t=-114
t=-115
t=-116
t=-117
t=-118
t=-119
t=-120
t=-121
t=-122
t=-123
t=-124
t=-125
t=-126
t=-127
t=-128
t=-129
t=-130
t=-131
t=-132
t=-133
t=-134
t=-135
t=-136
t=-137
t=-138
t=-139
t=-140
t=-141
t=-142
t=-143
t=-144
t=-145
t=-146
t=-147
t=-148
t=-149
t=-150
t=-151
t=-152
t=-153
t=-154
t=-155
t=-156
t=-157
t=-158
t=-159


In [7]:
# Calculate correlations
def calculate_correlation(x, y):
    return np.corrcoef(x, y)[0, 1]

pop0 = state.pop0
popminus5 = state.popminus5
popminus10 = state.popminus10
earth_indices = state.earth_indices

print('CORRELATIONS WITH 1995 DATA - CELL LEVEL')
print(calculate_correlation(popminus5[earth_indices], H0_arr[earth_indices] * l_b[:, 4]))
print(calculate_correlation(np.log(popminus5[earth_indices]), np.log(H0_arr[earth_indices] * l_b[:, 4])))
print(calculate_correlation(pop0[earth_indices] - popminus5[earth_indices], pop0[earth_indices] - H0_arr[earth_indices] * l_b[:, 4]))
print(calculate_correlation(np.log(pop0[earth_indices]) - np.log(popminus5[earth_indices]), np.log(pop0[earth_indices]) - np.log(H0_arr[earth_indices] * l_b[:, 4])))

print('CORRELATIONS WITH 1990 DATA - CELL LEVEL')
print(calculate_correlation(popminus10[earth_indices], H0_arr[earth_indices] * l_b[:, 9]))
print(calculate_correlation(np.log(popminus10[earth_indices]), np.log(H0_arr[earth_indices] * l_b[:, 9])))
print(calculate_correlation(pop0[earth_indices] - popminus10[earth_indices], pop0[earth_indices] - H0_arr[earth_indices] * l_b[:, 9]))
print(calculate_correlation(np.log(pop0[earth_indices]) - np.log(popminus10[earth_indices]), np.log(pop0[earth_indices]) - np.log(H0_arr[earth_indices] * l_b[:, 9])))

print('CORRELATIONS WITH 1995 DATA - COUNTRY LEVEL')
ctry_idx = state.C_vect.astype(int) - 1
popminus5_ctry_d = np.bincount(ctry_idx, weights=popminus5[earth_indices])
popminus5_ctry_m = np.bincount(ctry_idx, weights=H0_arr[earth_indices] * l_b[:, 4])
pop0_ctry = np.bincount(ctry_idx, weights=pop0[earth_indices])
print(calculate_correlation(popminus5_ctry_d, popminus5_ctry_m))
print(calculate_correlation(np.log(popminus5_ctry_d), np.log(popminus5_ctry_m)))
print(calculate_correlation(pop0_ctry - popminus5_ctry_d, pop0_ctry - popminus5_ctry_m))
print(calculate_correlation(np.log(pop0_ctry) - np.log(popminus5_ctry_d), np.log(pop0_ctry) - np.log(popminus5_ctry_m)))

print('CORRELATIONS WITH 1990 DATA - COUNTRY LEVEL')
popminus10_ctry_d = np.bincount(ctry_idx, weights=popminus10[earth_indices])
popminus10_ctry_m = np.bincount(ctry_idx, weights=H0_arr[earth_indices] * l_b[:, 9])
print(calculate_correlation(popminus10_ctry_d, popminus10_ctry_m))
print(calculate_correlation(np.log(popminus10_ctry_d), np.log(popminus10_ctry_m)))
print(calculate_correlation(pop0_ctry - popminus10_ctry_d, pop0_ctry - popminus10_ctry_m))
print(calculate_correlation(np.log(pop0_ctry) - np.log(popminus10_ctry_d), np.log(pop0_ctry) - np.log(popminus10_ctry_m)))

# Save all the output to disk
Path(os.path.join(model_dir, 'Output')).mkdir(parents=True, exist_ok=True)
sio.savemat(os.path.join(model_dir, 'Output/realgdp_w.mat'), {'realgdp_w': realgdp_w})
sio.savemat(os.path.join(model_dir, 'Output/u_w.mat'), {'u_w': u_w})
sio.savemat(os.path.join(model_dir, 'Output/u2_w.mat'), {'u2_w': u2_w})
sio.savemat(os.path.join(model_dir, 'Output/prod_w.mat'), {'prod_w': prod_w})
sio.savemat(os.path.join(model_dir, 'Output/phi_w.mat'), {'phi_w': phi_w})
sio.savemat(os.path.join(model_dir, 'Output/PDV_u_w.mat'), {'PDV_u_w': PDV_u_w})
sio.savemat(os.path.join(model_dir, 'Output/PDV_u2_w.mat'), {'PDV_u2_w': PDV_u2_w})
sio.savemat(os.path.join(model_dir, 'Output/PDV_realgdp_w.mat'), {'PDV_realgdp_w': PDV_realgdp_w})
sio.savemat(os.path.join(model_dir, 'Output/migr_cell.mat'), {'migr_cell': migr_cell})
sio.savemat(os.path.join(model_dir, 'Output/migr_ctry.mat'), {'migr_ctry': migr_ctry})
sio.savemat(os.path.join(model_dir, 'Output/l.mat'), {'l': l})
sio.savemat(os.path.join(model_dir, 'Output/u.mat'), {'u': u})
sio.savemat(os.path.join(model_dir, 'Output/realgdp.mat'), {'realgdp': realgdp})
sio.savemat(os.path.join(model_dir, 'Output/tau.mat'), {'tau': tau})
sio.savemat(os.path.join(model_dir, 'Output/l_b.mat'), {'l_b': l_b})

CORRELATIONS WITH 1995 DATA - CELL LEVEL
-0.00848382623533931
0.29363226375006907
0.055903475333739086
0.13790615004632448
CORRELATIONS WITH 1990 DATA - CELL LEVEL
-0.009141672895315492
0.2937642857322502
0.045179668284390365
0.07766881153775144
CORRELATIONS WITH 1995 DATA - COUNTRY LEVEL
0.1304051867099383
0.4720900089259972
0.013566161669815334
0.2763251335881252
CORRELATIONS WITH 1990 DATA - COUNTRY LEVEL
0.1332214926937118
0.47290323066812
0.02434678285162321
0.32394183306091895


In [6]:
# Free CMA buffers
state.kernel.close()